# 06 · Write-up
**Brief: Task 4 ("Write up your findings").** This notebook is meant to be read **on its own**. It gives the
answer, defines every term, explains how the project fits together, and summarises the key conclusions of
notebooks 01–05, so you only need to open those if you want the detail. It then covers what the brief asks
for in the write-up: the shortlist, the rejected candidates and why, assumptions, data source, and design
choices.

**How to read it:** section 1 is the answer; sections 2–3 are the background you need; section 4 is a
notebook-by-notebook tour; sections 6 onwards are the write-up proper.

### What this notebook answers from the brief
| Brief asks for | Where |
|---|---|
| Your shortlist of surviving precursors | sections 1 and 6 (each rule written out in full) |
| The candidates you rejected and why | section 7 |
| A brief description of your assumptions | section 9 (incl. every filter used to score the holdout machines, and why alerts after a repair are muted) |
| The data source, with enough detail to reproduce | section 10 |
| Your main design choices (H, W, thresholds, how you defined normal) | section 11 (incl. why H = 12, W = 36, with precedent) |
| If you run out of time: what you'd do next | section 12 |
| If you used AI tools: transcripts and notes on corrections | section 13 and the README |
| Background: build a small version of Oden's root-cause workflow | section 5 |
| (also) Why the simple rules beat an ML model | section 8 |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import json, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import pandas as pd
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- The saved results of notebooks 01-05 ------------------------------------------------
# This notebook computes nothing: it reads what notebooks 01-05 saved to outputs/. Run them first, in order.
from evaluation import load_results, check_results   # src/evaluation.py §9
display(check_results())                              # when each notebook ran, and whether it is still up to date

N1 = load_results("01_dataset_and_parameters")        # dataset profile, timestamp audit, look-alikes
N2 = load_results("02_candidates")                    # the ten candidates
N3 = load_results("03_ml_benchmark")                  # the ML models' scores
N4 = load_results("04_test_and_select")               # every candidate's scores, gates and verdict; the H x W grid
N5 = load_results("05_audit")                         # the audits
K = N4["kinds"]["any"]                                # "any failure", the main event type

,ran_at,H,W,status
notebook,,,,
01_dataset_and_parameters,2026-11-02 02:46:24,0,48,up to date
02_candidates,2026-11-02 02:46:28,0,48,up to date
03_ml_benchmark,2026-11-02 02:48:04,0,48,up to date
04_test_and_select,2026-11-02 02:57:24,0,48,up to date
05_audit,2026-11-02 02:57:38,0,48,up to date


## 1 · The answer in one paragraph
On the Azure Predictive Maintenance data (719 failure events, hourly readings, 100 machines, all of 2015),
each part's own sensor signal shifts about **48 hours** before it fails. But each part's signal also shifts
~470 times a year *without* a failure, so a sensor alone can't be trusted (at most ~38% of its alarms are real).
What separates real from false shifts is **the part's own error code**, logged 24 hours before the failure.
Alerts are scored on a window of **H = 12, W = 36**: an alert counts if it comes between 48 and 12 hours before a
failure, so every alert that counts leaves at least half a day to act. The strongest precursor, **C09 "matched
signature"** (a part's own signal drifting *and* its own error code logged), scores **F1 0.94 on machines it never
saw**: it catches **99%** of failures, **90%** of its alarms are real, and it warns about **24 hours** ahead. Two
more survive: **C10** (drift on a worn part) warns earlier (~43 h) but is right only 41% of the time, and **C08**
(any drift + any error) is a simpler fallback. The data is synthetic and unusually clean, so treat the absolute
numbers as optimistic and the method as the transferable result.

## 2 · Key terms
| Term | Meaning |
|---|---|
| **Event** | what we want to warn about: a component failure. Failures on the same machine at the same hour count as one event (the brief's rule). "Any failure" is the main target; each part (comp1–comp4) is also run on its own. |
| **Precursor / detector** | a pattern that tends to come before an event, written as a simple yes/no rule that is checked every hour ("alert" or "no alert"). |
| **H (lead time)** and **W (window)** | for a failure at time T, an alert counts if it falls between T − (H + W) and T − H. **H = 12, W = 36** (the window used here) means "between 48 and 12 hours before the failure". |
| **Normal operation** | hours far from any failure: not in any failure's window, not in the 24 h after a failure or repair, and not in the first 24 h of data. |
| **Threshold ("the line")** | the level a reading must cross to raise an alert, set from normal running so that normal hours almost never cross it (section 9, "How the alert lines are set"). |
| **Smoothing** | cleaning up a jittery reading before comparing it to the line, e.g. a **6 h average** (the mean of the last 6 hours). |
| **Variant** | one concrete version of an idea (e.g. "pressure, 6 h average, per-machine line at P99"). Each idea is tried in 20–95 variants, 655 in total. |
| **Lift** | how much more often a detector fires before failures than in normal operation (2 = twice as often). |
| **Precision** | of the alerts it raises, the share followed by a failure within H + W. |
| **Coverage** | of the failures, the share that had at least one alert in their window. |
| **F1** | the single score used to pick detectors: `2 × precision × coverage / (precision + coverage)`. It is high only when *both* are high, so it can't be gamed by never firing (which inflates lift) or always firing (which inflates coverage). |
| **Dev / holdout machines** | the 100 machines are split 50/50. Every choice is made on the **dev** machines; every reported number comes from the **holdout** machines, which never influenced a choice. |
| **Gate** | a pass/fail test that a candidate must pass before it can be kept, however good its F1. See below. |
| **Look-alike drift** | a signal shift that looks exactly like a pre-failure shift but isn't followed by a failure (~470 per part per year). |
| **Cluster bootstrap** | how the 95% intervals are made: resample whole holdout machines, so failures on the same machine aren't treated as independent. |

### What a "gate" is
Picking the best-scoring idea isn't enough: with 655 variants tried, some will score well by luck or by
accidentally using information from the future. So before ranking, every idea must pass **four gates**,
all checked on dev machines:

1. **Not luck (null test).** Each machine's alert timeline is shifted in time by a random amount, 500 times.
   That keeps how often and how burstily the detector fires, but breaks any real link to failures. The real
   detector must beat these shuffled copies by a wide margin. The bar is made stricter because we test ten
   ideas (a Bonferroni correction: p < 0.05 / 10 = 0.005).
2. **Lift ≥ 2.** It must fire at least twice as often before failures as in normal operation.
3. **Coverage ≥ 5%.** It must catch a meaningful share of failures.
4. **Causal check.** Its thresholds are re-set using only January–June, then it is scored on July–December.
   It must keep at least half its F1. A rule whose skill quietly depends on the future collapses here.

Ideas that pass all four are ranked by dev F1, and the top three are kept, skipping any that is the **same
alarm** as a better one (sharing ≥ 80% of its firing hours). Nine of the ten ideas passed every gate; only
C07 (overdue maintenance) failed, on the causal check.

## 3 · How the project fits together, and where the numbers come from
**Two kinds of files:**
* **`src/` (three helper files)** hold every function: `data.py` (settings, loading, windows), `detectors.py`
  (the rules and the ten candidates), `evaluation.py` (metrics, the selection protocol, audits, ML benchmark,
  and saving results).
* **Notebooks 01–06.** Run them **in order**. Each of 01–05 computes its own analysis live from the raw data
  and, in its last cell, saves its key results to `outputs/`:

| Run | Computes | Saves |
|---|---|---|
| 01 Dataset and parameters | the dataset profile, the timestamp audit, look-alike drifts, confidence | `outputs/01_dataset_and_parameters.json` |
| 02 Candidates | the ten candidates and their variants | `outputs/02_candidates.json` |
| 03 ML benchmark (~2 min) | three ML models, trained and scored | `outputs/03_ml_benchmark.json` |
| 04 Test and select (~10 min) | the full selection protocol for all five event types, and the H × W grid | `outputs/04_test_and_select.json` |
| 05 Audit | the audits of the detectors 04 kept (it reads 04's file for which ones) | `outputs/05_audit.json` |
| 06 Write-up (this one) | nothing: it reads the five files above | — |

**Why save results at all?** So this write-up can summarise everything without re-running ~15 minutes of
analysis, and so every number quoted here comes from the notebook that produced it. Each file records the
settings it was made with (H, W, seed) and the time it ran.

**Guarding against out-of-date files.** The table at the top of this notebook lists each file with the time
it ran. It says "re-run" if the code in `src/` changed afterwards, or if notebook 04 was re-run after notebook
05 (05 audits 04's detectors), and it warns if the notebooks used different H and W settings.

## 4 · What each notebook found

Each subsection gives the notebook's key conclusions, then the saved numbers behind them.

### 4.1 · Notebook 01: dataset and parameters
| Characteristic | Finding |
|---|---|
| Events | 761 failure rows → **719 events** (same machine + hour merged); 131–259 per part |
| Sampling and gaps | hourly, 100 machines × 8,761 h (all of 2015); no gaps, no missing values |
| Clustering | only 3 events come within a week of the previous one on the same machine; failures cluster *by machine* (0–19 each) |
| Operating regimes | effectively one: the four machine models run at the same signal levels, and machines never stop |
| Counters vs. live readings | the four sensors (volt, rotate, pressure, vibration) are live readings; the only counter is "hours since maintenance", which we build from the maintenance log |
| Timestamps | failures are logged in a daily 06:00 batch, but an hour-by-hour check finds no post-failure data before the stamp |
| Where the warning signs are | each part has one signature signal (comp1 → volt, comp2 → rotate, comp3 → pressure, comp4 → vibration). It shifts ~48 h before failure; the part's own error code lands exactly 24 h before |
| Key property | ~470 look-alike drifts per part never end in a failure, capping a signal-only rule at 21–34% precision; the error code is in 96–100% of real drifts but only 14–18% of look-alikes |
| Confidence | 363 holdout events for "any failure" (±5 points), 65 for comp3 (±12); the data is synthetic, so treat absolute numbers as optimistic |

In [3]:
# Saved by notebook 01: the dataset in numbers, and the look-alike drifts behind the key property.
e, g = N1["events"], N1["gaps"]
print(f"{e['failure_rows']} failure rows -> {e['events']} events ({e['multi_part']} with 2+ parts); per part: {e['per_part']}")
print(f"{N1['sampling']['machines']} machines x {N1['sampling']['hours']:,} hourly readings, {N1['sampling']['start'][:10]} to {N1['sampling']['end'][:10]}")
print(f"median {g['median_days_between_events']:.0f} days between events on a machine; {g['events_within_7_days']} within a week")
pd.DataFrame(N1["lookalikes"]).set_index("index")

761 failure rows -> 719 events (42 with 2+ parts); per part: {'comp1': 192, 'comp2': 259, 'comp3': 131, 'comp4': 179}
100 machines x 8,761 hourly readings, 2015-01-01 to 2016-01-01
median 30 days between events on a machine; 3 within a week


,signal,episodes_ending_in_failure,episodes_without_failure,precision_ceiling,share_with_error_failure,share_with_error_none,precision_ceiling_with_error
index,,,,,,,
comp1,volt,172,464,0.270,0.959,0.144,0.711
comp2,rotate,248,475,0.343,0.992,0.183,0.739
comp3,pressure,126,483,0.207,1.000,0.153,0.630
comp4,vibration,171,476,0.264,1.000,0.153,0.701


### 4.2 · Notebook 02: candidate precursors
Ten candidates in four families, each with a physical story and a yes/no detector (table below).

Ideas came from domain knowledge (bearing wear, friction, blockage, electrical faults), from what notebook 01
found in the data, and from the ML models in notebook 03. Each idea is built in many variants (655 in total),
so ideas are compared fairly rather than on a lucky first draft.

In [4]:
# Saved by notebook 02: the ten candidates.
pd.DataFrame(N2["candidates"])[["id", "title", "family", "variants"]]

,id,title,family,variants
0,C01,Voltage creeps up,telemetry,95
1,C02,Rotation sags,telemetry,95
2,C03,Pressure builds,telemetry,95
3,C04,Vibration rises,telemetry,95
4,C05,Any signal drifts,combo,95
5,C06,Any error code,errors,20
6,C07,Overdue maintenance,counter,20
7,C08,Drift confirmed by an error,combo,40
8,C09,Matched signature (part's own signal + own error),combo,40
9,C10,Drift on an aging part,combo,60


### 4.3 · Notebook 03: ML benchmark
* Three models (gradient boosting with and without the maintenance counters, and logistic regression), trained
  on dev machines from 32 features and scored exactly like the rules. The best, logistic regression, reaches
  F1 ~0.63, well below C09; section 8 compares them and explains why the rules win.
* **What the models lean on:** smoothed signals (6 h and 24 h averages) and each part's own error counts, which
  pointed the candidates toward "smoothed drift + error code".

In [5]:
# Saved by notebook 03: each ML model on the holdout machines.
pd.DataFrame(N3["models"])[["detector", "F1", "F1_CI", "precision", "coverage"]].round(3)

,detector,F1,F1_CI,precision,coverage
0,ML: logistic regression,0.627,"[0.59, 0.66]",0.457,0.997
1,"ML: gradient boosting, no maintenance counters",0.518,"[0.49, 0.55]",0.349,1.000
2,"ML: gradient boosting, all features",0.306,"[0.27, 0.34]",0.181,1.000


### 4.4 · Notebook 04: test and select
* **All ten ideas** were scored with lift, precision and coverage, with thresholds from normal operation only.
  Nine passed every gate; **C07 failed the causal check**. Why the others lose: section 7.
* **Kept:** **C09**, **C08** and **C10** (section 6 has their scores and the rules in full).
* **Smoothing:** a 6 h average works best; CUSUM is worst.
* **Choosing H and W:** one F1 grid of 100 windows. F1 is flat for every window that contains the error code
  logged 24 h before a failure, and collapses once the window starts 36 h or more ahead. The window is
  **H = 12, W = 36** (section 11).
* **Every event type:** C09 ranks first for any failure and for each part separately.

In [6]:
# Saved by notebook 04: the shortlist for every event type, and the H x W grid's key numbers.
display(pd.DataFrame([dict(event=e, rank=k["kept"].index(c["cid"]) + 1, id=c["cid"], idea=c["title"],
                           F1=c["hold"]["f1"], precision=c["hold"]["precision"], coverage=c["hold"]["coverage"])
                      for e, k in N4["kinds"].items() for c in k["concepts"] if c["status"] == "kept"]
                     ).sort_values(["event", "rank"]).round(3))
G = pd.DataFrame(N4["hw_grid"]).set_index("H")
G.columns = G.columns.astype(int)
print(f"H x W grid (dev F1): chosen H=12, W=36 -> {G.loc[12, 36]:.3f} | best anywhere -> {G.values.max():.3f} | "
      f"windows starting >= 36 h ahead: at most {G.loc[[36, 48, 72]].values.max():.2f}")

,event,rank,id,idea,F1,precision,coverage
1,any,1,C09,Matched signature (part's own signal + own error),0.918,0.927,0.909
0,any,2,C08,Drift confirmed by an error,0.764,0.628,0.977
2,any,3,C10,Drift on an aging part,0.566,0.413,0.903
4,comp1,1,C09,Matched signature (part's own signal + own error),0.942,0.891,1.000
5,comp1,2,C10,Drift on an aging part,0.574,0.406,0.977
3,comp1,3,C01,Voltage creeps up,0.376,0.255,0.721
8,comp2,1,C09,Matched signature (part's own signal + own error),0.945,0.909,0.985
7,comp2,2,C08,Drift confirmed by an error,0.814,0.694,0.985
6,comp2,3,C06,Any error code,0.263,0.622,0.167
10,comp3,1,C09,Matched signature (part's own signal + own error),0.909,0.933,0.887


H x W grid (dev F1): chosen H=12, W=36 -> 0.936 | best anywhere -> 0.948 | windows starting >= 36 h ahead: at most 0.24


### 4.5 · Notebook 05: audit (trying to break the results)
| Trap | Result |
|---|---|
| Keeping the lucky ones | kept detectors sit far above the shuffled-timing null (z ≈ 42–53); the best of 300 pure-noise detectors scores F1 0.09 on dev and 0.07 on holdout |
| Leakage through the event hour | scoring the failure hour turns "a repair was logged" into a fake perfect detector (lift ~67,000); our windows never score it |
| Counters that leak the future | "hours since maintenance" shows huge lifts (600–2,000), then drops to F1 ≈ 0 with past-only thresholds; the causal gate rejects it |
| Time drift | thresholds from January–June work almost as well on July–December (C09 F1 0.87 vs 0.92; C08 and C10 unchanged) |
| Machine mode changes | C09 holds across every machine model and age band (precision 0.88–0.94, coverage 0.92–0.97) |
| Clustered events | too rare to matter (3 within a week); intervals resample whole machines anyway |

In [7]:
# Saved by notebook 05: the luck check, and thresholds from January-June scored on July-December.
L = N5["audits"]["lucky"]
print(f"best of {len(L['dev_f1'])} pure-noise detectors: dev F1 {L['best_dev']}, holdout F1 {L['best_hold']}")
pd.DataFrame([dict(id=t["cid"], F1_full_year_thresholds=t["full_year_thresholds"]["f1"],
                   F1_Jan_Jun_thresholds=t["first_half_thresholds"]["f1"]) for t in N5["audits"]["temporal"]]).round(3)

best of 300 pure-noise detectors: dev F1 0.0861, holdout F1 0.071


,id,F1_full_year_thresholds,F1_Jan_Jun_thresholds
0,C08,0.749,0.754
1,C09,0.920,0.873
2,C10,0.550,0.571


## 5 · How this mirrors Oden's Agentic Root Cause Analysis workflow
The brief asks for "a small version" of the workflow in Oden's post,
[Agentic Root Cause Analysis](https://engineering.oden.io/blog/agentic-root-cause-analysis). Each of its steps
has a counterpart here:

| Oden's workflow | This project |
|---|---|
| **1. Build operating context** (line diagrams, process knowledge, operator interviews) | Notebook 01: EDA of the five raw files, the dataset profile, and an audit of how failures are timestamped |
| **2. Reduce the search space with ML** (gradient-boosted classifiers, feature importances) | A gradient-boosting / logistic-regression benchmark and its feature importances, used for ideas, not as a detector (**notebook 03**) |
| **3. Generate hypotheses** that tie a signal pattern to a physical mechanism and a testable statistic | Notebook 02: 10 candidates, each with a story and a fixed yes/no detector |
| **4. Filter by lift analysis**: lift, precision, and permutation tests against random nominal windows | Notebook 04: lift, precision and coverage for every candidate; a circular-shift null test (a permutation test that keeps each detector's firing pattern); thresholds from nominal ("normal") running only, with downtime-like periods masked |
| **5. Root-cause families, playbooks, alerts** | The shortlist forms one family per part: C09 (own signal + own error code) is the "act now" alarm, and C10 (drift on an aging part) the earlier watchlist. Playbooks would come next, with plant input |

Two points from the post that this data echoes:
* **Lift and precision understate real causes**, partly because operators sometimes prevent the event. Here, ~470
  look-alike excursions per part never end in a failure (notebook 01, section 7). Some may be failures that were
  averted. That caps the precision of any signal-only rule and is why the error-code confirmation matters.
* **Atomic, explainable detectors are preferred.** C09 is one pattern applied to each part. Per component it is a
  single two-condition rule (own signal drifting AND own error code logged), as the brief allows: "a candidate can
  use one signal or combine several".

## 6 · Shortlist: the surviving precursors ("any failure", holdout machines, H = 12, W = 36)

In [8]:
# Saved by notebook 04 (section 7): the three kept rules at H=12, W=36, on holdout machines.
DEP = N4["deployment"]["rules"]
pd.DataFrame([dict(id=cid, precursor=next(c["title"] for c in K["concepts"] if c["cid"] == cid),
                   F1=d["retuned"]["f1"], precision=d["retuned"]["precision"], coverage=d["retuned"]["coverage"],
                   lift=d["retuned"]["lift"], lead_h=d["retuned"]["lead_median"],
                   alerts_per_machine_month=d["retuned"]["alerts_per_machine_month"]) for cid, d in DEP.items()]).round(3)

,id,precursor,F1,precision,coverage,lift,lead_h,alerts_per_machine_month
0,C08,Drift confirmed by an error,0.744,0.628,0.914,27.959,24.0,2.192
1,C09,Matched signature (part's own signal + own error),0.941,0.898,0.989,236.379,24.0,1.478
2,C10,Drift on an aging part,0.564,0.413,0.889,16.108,43.0,4.131


**Conclusion.** Three precursors survive. **C09** (a part's own signal drifting *and* its own error code) is
the alarm to act on: 90% precision, 99% coverage, ~1.5 alerts per machine per month, ~24 h ahead. **C10** (drift
on an aging part) is an earlier, noisier watchlist (~43 h ahead, 41% precision). **C08** (any drift + any error)
is the fallback that needs no part ↔ code pairing.

### The three rules, written out
Each rule is checked **every hour on every machine**. Shared definitions:
* **6 h average**: the mean of the sensor's last 6 hourly readings.
* **Robust spread** of a sensor: 1.4826 × the median absolute deviation of its 6 h average, from that machine's
  own readings (normal-operation hours for dev machines; all of its hours for holdout machines).
* **Each part's own sensor and error code(s)**, learned from the dev machines:

| Part | Sensor (direction of a fault) | Own error code(s) |
|---|---|---|
| comp1 | voltage (rises) | error1 |
| comp2 | rotation (falls) | error2 or error3 |
| comp3 | pressure (rises) | error4 |
| comp4 | vibration (rises) | error5 |

**C09 · Matched signature** (the alarm to act on). Raise an alert when, **for any one part**, both hold:
1. that part's sensor (6 h average) is **more than 3.09 robust spreads** from the machine's median, in the fault
   direction (the machine's P99.9 level). On a typical machine: voltage above ~190 (normal ~170), rotation
   below ~383 (normal ~449), pressure above ~113 (normal ~100), vibration above ~47 (normal ~40); **and**
2. that part's **own error code** has been logged **at least once in the last 24 hours**.

**C08 · Drift + any error** (the fallback). Raise an alert when both hold:
1. **any** of the four sensors (6 h average) is **more than 3.72 robust spreads** from the machine's median, in
   its fault direction (P99.99: on a typical machine voltage above ~194, rotation below ~370, pressure above
   ~116, vibration above ~48); **and**
2. **any** error code has been logged **at least once in the last 48 hours**.

**C10 · Drift on an aging part** (the early warning). Raise an alert when, **for any one part**, both hold:
1. that part's sensor (6 h average) is **more than 3.72 robust spreads** from the machine's median, in the fault
   direction (the same level as C08); **and**
2. that part has gone **longer since its last replacement than its median in normal running**: about 33 days
   for comp1, 31 for comp2, 34 for comp3 and 33 for comp4.

An alert is "on" for as long as the conditions hold; a new alert starts each time it switches from off to on.

## 7 · Rejected candidates, and why

In [9]:
# Every idea that was not kept, with the protocol's reason (which gate it failed, or what outranked it).
pd.DataFrame([dict(id=c["cid"], idea=c["title"], hold_F1=c["hold"]["f1"], reason=c["reason"])
              for c in K["concepts"] if c["status"] != "kept"]).round(3)

,id,idea,hold_F1,reason
0,C01,Voltage creeps up,0.283,"passed all gates but outranked by three stronger, non-redundant detectors"
1,C02,Rotation sags,0.390,"passed all gates but outranked by three stronger, non-redundant detectors"
2,C03,Pressure builds,0.246,"passed all gates but outranked by three stronger, non-redundant detectors"
3,C04,Vibration rises,0.288,"passed all gates but outranked by three stronger, non-redundant detectors"
4,C05,Any signal drifts,0.441,"passed all gates but outranked by three stronger, non-redundant detectors"
5,C06,Any error code,0.566,"passed all gates but outranked by three stronger, non-redundant detectors"
6,C07,Overdue maintenance,0.106,collapses with past-only thresholds (Jul–Dec F1 0.00 vs 0.05 with full-year thresholds): its skill leaks from the fu...


**Conclusion.** Six of the seven rejected ideas are real but weaker signals that were outranked: single signals
(C01–C04) are fooled by look-alike drifts, "any signal drifts" (C05) raises too many false alarms, and "any error
code" (C06) misses half the failures. Only **C07 (overdue maintenance)** failed a gate: its skill leaks from the
future (notebook 05, section 3).

## 8 · Why the simple rules win
Notebook 03 trained three ML models on every feature the rules could use (32 per machine-hour), then scored
them exactly like the rules: same window, same alert threshold method (P99 of normal operation), same holdout
machines. The table below puts them next to the three kept rules. (ML scores shift by about ±0.02 between
runs, because training uses a random sample of normal hours.)

In [10]:
# The three kept rules (notebook 04) next to the three ML models (notebook 03): same window, same holdout machines.
rules = [dict(detector=f"rule: {c['cid']} {c['title']}", F1=c["hold"]["f1"], precision=c["hold"]["precision"],
              coverage=c["hold"]["coverage"]) for c in K["concepts"] if c["status"] == "kept"]
ml = pd.DataFrame(N3["models"])[["detector", "F1", "precision", "coverage"]].to_dict("records")
pd.DataFrame(rules + ml).sort_values("F1", ascending=False).round(3)

,detector,F1,precision,coverage
1,rule: C09 Matched signature (part's own signal + own error),0.918,0.927,0.909
0,rule: C08 Drift confirmed by an error,0.764,0.628,0.977
3,ML: logistic regression,0.627,0.457,0.997
2,rule: C10 Drift on an aging part,0.566,0.413,0.903
4,"ML: gradient boosting, no maintenance counters",0.518,0.349,1.000
5,"ML: gradient boosting, all features",0.306,0.181,1.000


**Why a two-condition rule beats a 32-feature model here:**
* **The precursor has a sharp, specific structure:** *a part's own signal drifts while its own error code is
  logged*. C09 encodes that directly. A generic model trained on "any failure vs normal" has to discover the
  part-specific pairing from the data, and the linear model can't express it at all.
* **Look-alikes fool anything that scores signal levels.** Each part's signal drifts ~470 times a year without
  a failure (notebook 01, section 7). Only the matching error code separates real drifts from look-alikes, and
  the rule demands it explicitly.
* **The rules are tuned on the score we report** (alert-level F1). The models are trained to rank individual
  hours, then thresholded once, which is a weaker fit to the goal.
* **The models catch every failure but raise far more false alarms**, and gradient boosting with counters leans
  on "hours since maintenance", the same leaky signal the causal gate rejects (notebook 05, section 3).

**Limits:** no hyperparameter search, one alert threshold, and one "any failure" output rather than one per
part. This is a sanity ceiling, not the best possible ML. A rule that is also easier to explain and trust is the
better choice unless a model clearly beats it, and here none does.

## 9 · Assumptions
* **Failure timestamps mark the end of the degradation.** They are logged in a daily 06:00 batch, but an
  hour-by-hour audit (notebook 01, section 3) finds no post-failure data before them, so the stamp is treated
  as the failure time. Failure-linked error codes are stamped the same way.
* **The repair logged with each failure is part of the failure,** not a precursor. The failure hour itself is
  never scored.
* **Telemetry is a true hourly reading** at its timestamp. Every feature uses only past hours (causal).
* **Machines run continuously** (minimum rotation 138 rpm), so there is no idle mode to exclude.
* **Machines are exchangeable** enough that 50 dev machines can inform 50 holdout machines (split stratified
  by machine model).
* **Alerts in the 24 h after a repair or replacement are muted** (see "Why mute alerts after a repair" below).
* **The look-alike excursions** (signal drifts with no failure after them) are real behaviour of this data,
  perhaps averted failures or simulator noise. They are not relabelled.

### How the holdout machines are scored: every filter, stated
All reported numbers come from the **50 holdout machines** (half the fleet, split at random within each machine
model, seed 7). These are the filters and assumptions applied when scoring them:

| What | Rule | Why |
|---|---|---|
| Which machines | 50 holdout machines; they never influenced any choice of idea, variant, threshold, part pairing or H and W | an honest estimate of how the rules do on machines they haven't seen |
| Which events | failures on the same machine at the same hour are merged into one event; an event counts only if its window starts after the 24 h warm-up and has at least one scorable hour. **350 of the 363 holdout events** count | the brief's definition of an event; a window that can't be observed can't be scored |
| The failure hour | never scored | it contains the repair record, which would be a fake "precursor" (notebook 05, section 2) |
| After a failure or replacement | the next **24 h are muted**: alerts there don't count, and those hours are not "normal" | see below |
| Warm-up | the first 24 h of the year are not scored | see "Why ignore the warm-up and the end of the year?" below |
| End of the year | an alert in the last H + W hours is not judged | see below |
| Normal operation | every hour that is not in the H + W hours before any failure and not muted | the brief's definition; every line is set from dev machines' normal hours (holdout per-machine lines: see below) |
| Alert lines | sensor lines are set **per machine** from that machine's own readings (holdout machines: all of their hours, never their failure dates); error-count and part-age lines are **one line for the fleet**, set on dev machines only | see "How the alert lines are set" below |
| Error codes and part ages | taken as logged | assumed to be recorded at the hour they happen |

### Why ignore the warm-up and the end of the year?
Both are hours we can't score fairly, and both are small.
* **Warm-up (the first 24 hours of 2015).** The rules use 6 h and 24 h averages and 24–48 h error counts. In the
  first day those don't have enough history yet: a "24 h average" at 3 a.m. on 1 January is really a 3-hour
  average. Events whose window would start before the data begins can't be scored either. On the holdout machines
  that is **13 of 363 events**, all stamped 2 January 2015 (their 48 h window would start in 2014).
* **End of the year (the last H + W hours, 48 h = 0.55% of the year).** An alert there can't be judged: the data
  stops before we would know whether a failure followed. Counting it as a false alarm would wrongly lower
  precision, and counting it as correct would wrongly raise it, so it is left out (right-censoring). No failures
  are dropped; it only affects which alerts count toward precision.

### Why mute alerts in the 24 h after a repair, and is it reasonable?
**What it does.** For 24 hours after any failure or any part replacement (scheduled or as a repair), alerts don't
count, and those hours are left out of "normal operation". About 6% of all hours are muted this way.

**Why.**
* **The crew is already there.** Right after a repair, the machine has just been worked on. An alert in those hours
  would not tell anyone something new, and in practice operators silence alarms during maintenance and restart.
* **The readings aren't normal running.** A machine that has just had a part replaced is being restarted and
  re-settled. Treating those hours as "normal" would distort the thresholds, and counting them as warnings would
  reward detectors for reacting to the repair itself.
* **It removes a leak.** Failures and repairs are logged together, so post-repair behaviour is tied to the failure
  that just happened, not to the next one.

**Is it reasonable? Yes, and the results don't depend on it.**
* **It hides almost nothing.** Only **1 of the 719 failures** came within 24 hours of the previous failure on the
  same machine, so the mute almost never covers a real warning.
* **The exact length barely matters.** Notebook 05, section 4 re-scores the kept rules with 0, 6, 24 and 72 hours
  of muting: C09's precision moves only from 0.92 to 0.94, and its coverage stays 0.91.
* **The risk to watch** in a real plant is a bad repair that fails again within a day. If that matters, the mute
  can be shortened (the audit shows it costs little) or replaced by a separate "post-repair check" alarm.

### How the alert lines are set
Every rule has two conditions, and each condition compares a number to a line.

**1. The sensor condition uses a line per machine.** Machines differ a little in their normal levels (one
machine's voltage might hover around 168, another's around 172), so each machine gets its own line: its median
reading plus 3.09 (C09) or 3.72 (C08, C10) *robust spreads*. The robust spread measures how much the readings
normally wobble, and like the median it barely moves when a few hours are unusual. Where the readings come from:
* **Dev machines** (used to build and tune the rules): their **normal-operation hours**, i.e. leaving out the
  hours just before a failure and just after a repair. That needs their failure dates, which is fine on dev.
* **Holdout machines** (used only to score): **all of their own hours**. In deployment we wouldn't know in advance
  when a machine will fail, so we can't tell which of its hours were "normal"; scoring a holdout machine must not
  use its failure dates either. Because only ~4% of a machine's hours come just before a failure, and the median
  and robust spread shrug off a few unusual hours, the line ends up almost the same.

**2. The error-count and part-age conditions use one line for the whole fleet,** set from the dev machines only:
"at least one error code in the last 24 h" (C09) or "48 h" (C08), and "part older than its median age since
replacement, about 31–34 days" (C10). Holdout machines play no part in setting these lines.

## 10 · Data source and reproduction
* **Data:** [Microsoft Azure Predictive Maintenance on Kaggle](https://www.kaggle.com/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance),
  version 3, five CSVs. It downloads automatically with
  `kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance")`; no account needed.
* **Code:** three helper files in `src/`. Run notebooks 01–06 in order; 01–05 each save their results to
  `outputs/` for this notebook (section 3). Machine split: seed 7, stratified by model.

## 11 · Main design choices
| Choice | Value | Why |
|---|---|---|
| Event | any component failure; same machine + same timestamp = one event (also run per component) | as the brief defines it; toggled in `Config(event=...)` |
| **H** (lead time) | **12 h** | every alert that counts leaves at least half a day to act; see "Why H = 12, W = 36" below |
| **W** (window) | **36 h** | reaches back to where each part's signal starts drifting, 48 h before a failure, so the window covers the whole precursor |
| **Normal operation** | not within H+W before *any* failure; not within 24 h after any failure or repair; not in the first 24 h | the brief's definition, plus a recovery guardrail and a warm-up for trailing averages |
| **Thresholds** | sensor lines per machine (median + robust spreads); error-count and part-age lines one per fleet | chosen on dev machines' normal-operation hours, as the brief requires; a holdout machine's per-machine line comes from all its own hours so its failure dates are never used (section 9) |
| Selection | F1 of precision and coverage on dev; gates: null test, lift ≥ 2, coverage ≥ 5%, causal check | F1 can't be gamed by never firing or always firing; gates guard against luck and leakage |

### Why H = 12, W = 36
* **H = 12: half a day of warning.** With H = 0 an alert one hour before a failure would count as a success,
  which leaves no time to act. With H = 12, only alerts at least 12 hours ahead count.
* **W = 36: the whole precursor.** Each part's signal starts drifting about 48 hours before it fails, and its own
  error code is logged 24 hours before. A window from 48 to 12 hours before the failure contains both.
* **It costs nothing in detection.** On the F1 grid (notebook 04, section 6) the best window anywhere scores 0.948
  and H = 12, W = 36 scores 0.936, within the noise (~±0.02). A bigger buffer would not work: F1 collapses once the
  window starts 36 hours or more ahead.

**Why 12 hours? Precedent.** No standard prescribes a single number, but the principle and the unit are both
well established:
* **The warning must be longer than the time needed to act.** In reliability-centred maintenance this is the
  *net P-F interval*: the time between a fault becoming detectable (P) and the failure (F), minus the time needed
  to respond, must be "long enough to be of some use" ([Reliabilityweb, *RCM: On-Condition Task Interval
  Determination*](https://reliabilityweb.com/articles/entry/rcm-on-condition-task-interval-determination), after
  Moubray's RCM2). The international standard for machine prognostics, **ISO 13381-1** (*Condition monitoring and
  diagnostics of machines: Prognostics*, clause 5.3 "Setting alert, alarm and trip limits"), likewise bases alarm
  limits on the **maintenance lead time required**, including planning time and spare-part delivery
  ([ISO 13381-1:2004 preview](https://www.sis.se/api/document/preview/905442/)).
* **One shift is the natural unit of response.** Plants that run around the clock commonly work **12-hour
  shifts**, and longer shifts are favoured precisely because extended maintenance jobs can be finished within
  one crew's shift rather than handed over ([Power Engineering, *Making the shift: 12-hour vs. 8-hour*](https://www.power-eng.com/business/people/making-the-shift12-hour-vs-8-hour/)).
  A 12-hour lead means the crew on shift when the alert fires can diagnose the problem, get the part and do the
  repair, or hand a planned job to the next shift, before the failure.

The right number for a real line should come from its own repair times and part lead times; 12 hours is a
sensible default.

## 12 · What we would do next
* Ask whether the look-alike excursions precede maintenance that averted a failure. If so, some "false
  alarms" are really saves.
* Replace F1 with an expected-cost score once the cost of a false alarm and of a missed failure are known.
* Test how much lead time error codes really give with a non-batched error log. Here they land exactly 24 h
  before the failure (notebook 01, section 3).
* Train a per-part ML model (one output per component) and score it with the same alert-level metrics.
* Repeat the dev/holdout split over many seeds to measure how stable the selection is.